# Creating the model

In [ ]:
# imports
import data_loader as dl
from data_loader import FingersDataset
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

Import the dataset:

In [ ]:
# %pip install kagglehub
# import kagglehub

# Download latest version (NOTE: Filesize is about 509MB, files are .png)
# path = kagglehub.dataset_download("koryakinp/fingers")

# locate the dataset files
path = "/cs/cs153/data/fingers/versions/2"

print("Path to dataset files:", path)

Now that dataset imported, let's look at it:

In [ ]:
import os
from PIL import Image

import matplotlib.pyplot as plt

path_in = path + "/test"

# Get list of image files from the dataset path
image_files = [f for f in os.listdir(path_in) if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:6]

# Display images in a grid
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for idx, img_file in enumerate(image_files):
    img_path = os.path.join(path_in, img_file)
    img = Image.open(img_path)
    axes[idx].imshow(img)
    
    # Extract number and side from filename
    parts = img_file.split('_')
    if len(parts) > 1:
        num_side = parts[1].split('.')[0]  # e.g., '4L'
        num = num_side[:-1]
        side = num_side[-1]
        if side == 'L':
            side_str = 'Left'
        elif side == 'R':
            side_str = 'Right'
        else:
            side_str = side
        title = f"{num} fingers ({side_str})"
    else:
        title = img_file
    
    axes[idx].set_title(title)
    axes[idx].axis('off')

# Add a title to the figure
fig.suptitle("Finger Count Images from Dataset")

plt.tight_layout()
plt.show()

Now let's use the dataloader to prepare the dataset for training, validating and testing.

In [ ]:
train_path = path + "/train"
test_path = path + "/test"

We can create different transformations so that we can make training more robust by introducing more noise:

In [ ]:
# from torch.utils.data import random_split
from data_loader import FingersDataset
from torch.utils.data import Subset
import numpy as np

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, 
                            translate=(0.1, 0.1), 
                            scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = FingersDataset(train_path, transform=train_transform)
val_dataset   = FingersDataset(train_path, transform=val_transform)

indices = np.arange(len(train_dataset))
np.random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx, val_idx = indices[:split], indices[split:]

train_data = DataLoader(Subset(train_dataset, train_idx), batch_size=32, shuffle=True)
val_data   = DataLoader(Subset(val_dataset, val_idx), batch_size=32, shuffle=False)

Now let's get our pretrained model and begin transfer learning!

In [ ]:
def evaluate(model, val_loader, criterion, device):
    '''Validate the model on the validation set and return average loss and accuracy.'''
    
    model.eval()  # important

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
from torchvision import models
import torch
import torch.nn as nn
import torch.optim as optim

import ssl
import certifi

ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 6)

# -- MAYBE LATER: MobileNetV2, supposed to be better for real-time applications -- #
# model = models.mobilenet_v2(pretrained=True)
# model.classifier[1] = nn.Linear(model.last_channel, 5)

# Now begin transfer learning!

# Assuming train_data is a DataLoader from dl.load_data
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# Move model to device (CPU or GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Training loop (simple example for transfer learning)
training_loss = []
validation_loss = []
num_epochs = 6
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_data:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    training_loss.append(running_loss / len(train_data))
    val_loss, val_acc = evaluate(model, val_data, criterion, device)
    validation_loss.append(val_loss)

    
    print(f'Epoch {epoch+1}/{num_epochs}, Training Loss: {running_loss/len(train_data):.4f}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}')

Then we can plot the training and validation losses:

In [ ]:
plt.plot(training_loss, label='Training Loss')
plt.plot(validation_loss, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

Testing the data and its accuracy in its prediction:

In [ ]:
test_data = dl.get_data_loader(test_path)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_data:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test Accuracy: {100 * correct / total:.2f}%')

# Processing the video and predicting classes based on model

First, we need to process the video that we want to use to test whether our model with customized classes will work. Using opencv, we can load the video frame-by-frame and save it as a greyscale image which aligns with the dataset that we are working with.

In [ ]:
import cv2
import os

cap = cv2.VideoCapture('vids/test.mp4')
actual_fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Actual FPS: {actual_fps}")

outfile = 'vids/test_gray/'
frame_count = 0

if not os.path.exists(outfile):
    os.makedirs(outfile)

while cap.isOpened():
    ret, frame = cap.read()
    if ret:
        frame_grey = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        cv2.imwrite(outfile + f'frame_{frame_count}.jpg', frame_grey)
        frame_count += 1
    else:
        print("Could not read frame.")
        break
else:
    print("Error: Could not open video.")

cap.release()

Then we can predict each frame:

In [ ]:
frames_path = "vids/test_gray/"

def get_frame_number(fname):
    return int(fname.split('_')[1].split('.')[0])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

pred_classes = []
with torch.no_grad():
    for frame_file in sorted(os.listdir(frames_path), key=get_frame_number):
        img_path = os.path.join(frames_path, frame_file)
        img = Image.open(img_path).convert('RGB')
        img_tensor = transform(img).unsqueeze(0).to(device)
        
        output = model(img_tensor)
        _, predicted = torch.max(output.data, 1)
        
        pred_classes.append(predicted.item())
        print(f"{frame_file}: Predicted {predicted.item()} fingers")
# print(sorted(os.listdir(frames_path), key=get_frame_number))

Music aspect now; We willl create the pure-tone chords that will be toggled by the predicted classes!

In [ ]:
def generate_tone(freq, t, sample_rate=44100):
    '''Generate a sine wave tone of a given frequency and duration.'''
    t = np.linspace(0, t, int(sample_rate * t), endpoint=False)
    tone = 0.5 * np.sin(2 * np.pi * freq * t)
    return tone

notes = {
    0: 261.63,  # C4
    1: 293.66,  # D4
    2: 329.63,  # E4
    3: 349.23,  # F4
    4: 392.00,  # G4
    5: 440.00,   # A4
    6: 493.88   # B4
}

import numpy as np

def generate_chords(predicted_class, duration=1.0, sample_rate=44100):
    '''Generate a chord based on the predicted number of fingers.'''
    
    # Define the chords based on your requirements
    # Format: {class_index: [list of frequencies]}
    chord_map = {
        0: [261.63, 329.63, 392.00],  # C Major (C4, E4, G4)
        1: [293.66, 349.23, 440.00],  # D Minor (D4, F4, A4)
        2: [329.63, 392.00, 493.88],  # E Minor (E4, G4, B4)
        3: [349.23, 440.00, 523.25],  # F Major (F4, A4, C5)
        4: [392.00, 493.88, 587.33],  # G Major (G4, B4, D5)
        5: [440.00, 523.25, 659.25],  # A Minor (A4, C5, E5)
        6: [0.0]                      # None class (Silence)
    }
    
    # Get the frequencies for the predicted class
    freqs = chord_map.get(predicted_class, [0.0])
    
    # Initialize an empty array for the total chord
    t_axis = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    chord_wave = np.zeros_like(t_axis)
    
    # Sum the sine waves for each frequency
    for freq in freqs:
        if freq > 0:
            # 0.3 factor keeps the combined volume from hitting the ceiling
            chord_wave += 0.3 * np.sin(2 * np.pi * freq * t_axis)
            
    return chord_wave

chord = generate_chords(4)


In [ ]:
import IPython.display as ipd
import librosa as lb
import soundfile as sf

fps = 30
total_frames = len(pred_classes)
total_duration = total_frames / fps

y_final = np.zeros(int(total_duration * 44100))
for i, pred in enumerate(pred_classes):
    chord = generate_chords(pred, duration=1/fps)
    start_idx = int(i * (44100 / fps))
    end_idx = start_idx + len(chord)
    y_final[start_idx:end_idx] += chord
ipd.Audio(y_final, rate=44100)
sf.write('output_chord.wav', y_final, 44100)

In [5]:
from moviepy import VideoFileClip, AudioFileClip

# 1. Load the video
video_clip = VideoFileClip("vids/test.mp4")

# 2. Load your generated audio
audio_clip = AudioFileClip("output_chord.wav")

# 3. Attach audio to video
final_video = video_clip.with_audio(audio_clip) # .with_audio() is the v2.0 way

# 4. Save the result
final_video.write_videofile("final_hand_music.mp4", codec="libx264")

MoviePy - Building video final_hand_music.mp4.
MoviePy - Writing audio in final_hand_musicTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video final_hand_music.mp4



MoviePy - Done !
MoviePy - video ready final_hand_music.mp4
